In [0]:
%sql
CREATE CATALOG IF NOT EXISTS hr_catalog;

CREATE SCHEMA IF NOT EXISTS hr_catalog.bronze;

CREATE SCHEMA IF NOT EXISTS hr_catalog.silver;

CREATE SCHEMA IF NOT EXISTS hr_catalog.gold;

CREATE VOLUME IF NOT EXISTS hr_catalog.bronze.hr_files;

In [0]:
#check the uploaded file 
display(dbutils.fs.ls("/Volumes/hr_catalog/bronze/hr_files/"))

In [0]:
#load the dataset using pandas 
import pandas as pd
import numpy as np

file_path = "/Volumes/hr_catalog/bronze/hr_files/erroneous_dataset.csv"

df = pd.read_csv(file_path, sep=",", dtype=str)

print("Dataset loaded successfully!")
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])

display(df.head())

In [0]:
# check columns name
print(df.columns.tolist())

In [0]:
#check missing values

# Convert empty spaces into missing values
df = df.replace(r"^\s*$", np.nan, regex=True)

# Count missing values in each column
missing_values = df.isnull().sum()

# Display only columns having missing values
missing_report = missing_values[missing_values > 0]

print("Columns containing missing values:")
display(missing_report.reset_index().rename(
    columns={"index": "Column", 0: "Missing_Count"}
))

In [0]:
#check duplicate employee records
duplicate_records = df[
    df.duplicated(subset=["Employee_ID"], keep=False)
]

print("Total duplicate records:", len(duplicate_records))

display(duplicate_records.head(10))

In [0]:
#understanding the duplicate record
# Count only extra duplicate records
extra_duplicates = df.duplicated(
    subset=["Employee_ID"],
    keep="first"
).sum()

print("Extra duplicate records:", extra_duplicates)

# Count Employee IDs appearing more than once
duplicate_id_counts = (
    df["Employee_ID"]
    .value_counts()
)

print(
    "Employee IDs appearing multiple times:",
    (duplicate_id_counts > 1).sum()
)

# Show the most repeated Employee IDs
display(
    duplicate_id_counts[duplicate_id_counts > 1]
    .head(10)
)

In [0]:
#Identify conflicting employee records

# Analyze repeated Employee IDs
duplicate_analysis = (
    df[df.duplicated("Employee_ID", keep=False)]
    .groupby("Employee_ID")
    .agg(
        Record_Count=("Employee_ID", "size"),
        Different_Names=("Employee_Name", "nunique"),
        Different_Departments=("Department", "nunique"),
        Different_Designations=("Designation", "nunique")
    )
    .reset_index()
)

print("Duplicate ID analysis:")
display(duplicate_analysis.head(10))

# Count IDs with conflicting employee details
conflicts = duplicate_analysis[
    (duplicate_analysis["Different_Names"] > 1) |
    (duplicate_analysis["Different_Departments"] > 1) |
    (duplicate_analysis["Different_Designations"] > 1)
]

print("IDs with conflicting employee details:", len(conflicts))

In [0]:
#Step 9: Check incorrect salary, age and experience
# # Convert columns into numeric values for validation
age = pd.to_numeric(df["Age"], errors="coerce")

experience = pd.to_numeric(
    df["Experience_Years"], errors="coerce"
)

salary = pd.to_numeric(
    df["Annual_Salary"].str.replace(
        r"[₹,$\s]", "", regex=True
    ),
    errors="coerce"
)

# Check invalid values
invalid_age = age.isna() | (age <= 0)

invalid_experience = (
    experience.isna() |
    (experience < 0) |
    (experience > age - 18)
)

invalid_salary = salary.isna() | (salary <= 0)

print("Invalid Age Records:", invalid_age.sum())
print("Invalid Experience Records:", invalid_experience.sum())
print("Invalid Salary Records:", invalid_salary.sum())

In [0]:
#Step 10: Check employment status and dates
# Check employment status
print("Employment Status Distribution:")
display(df["Employment_Status"].value_counts(dropna=False))

# Check joining dates
joining_dates = pd.to_datetime(
    df["Joining_Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

# Check appraisal dates
appraisal_dates = pd.to_datetime(
    df["Last_Appraisal"],
    format="%d-%m-%Y",
    errors="coerce"
)

print("Invalid Joining Dates:", joining_dates.isna().sum())
print("Invalid Appraisal Dates:", appraisal_dates.isna().sum())

# Check future joining dates
print(
    "Future Joining Dates:",
    (joining_dates > pd.Timestamp.today().normalize()).sum()
)

In [0]:
#Inspect the actual date format
# Display sample joining dates
print("Joining Date Samples:")
display(df["Joining_Date"].head(15))

# Display sample appraisal dates
print("Last Appraisal Samples:")
display(df["Last_Appraisal"].head(15))

# Check different date formats/values
print("Joining Date Examples:")
display(df["Joining_Date"].value_counts().head(10))

In [0]:
#Correct date validation
# Convert dates using the correct format
joining_dates = pd.to_datetime(
    df["Joining_Date"],
    format="%Y-%m-%d",
    errors="coerce"
)

appraisal_dates = pd.to_datetime(
    df["Last_Appraisal"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Validate dates
today = pd.Timestamp.today().normalize()

print("Invalid Joining Dates:", joining_dates.isna().sum())
print("Invalid Appraisal Dates:", appraisal_dates.isna().sum())

print(
    "Future Joining Dates:",
    (joining_dates > today).sum()
)

print(
    "Future Appraisal Dates:",
    (appraisal_dates > today).sum()
)

In [0]:
#Inspect future joining dates
# Find employees with future joining dates
future_joining = df.loc[
    joining_dates > today,
    ["Employee_ID", "Employee_Name",
     "Department", "Joining_Date", "Employment_Status"]
]

print("Future Joining Date Records:", len(future_joining))

display(future_joining.head(15))

In [0]:
#Step 14: Create a future joining date flag

# Flag employees having future joining dates

df["Invalid_Joining_Date"] = (
    joining_dates > today
)

print(
    "Future Joining Date Records:",
    df["Invalid_Joining_Date"].sum()
)

# Display flagged employees
display(
    df.loc[
        df["Invalid_Joining_Date"],
        ["Employee_ID", "Employee_Name",
         "Joining_Date", "Employment_Status"]
    ].head(10)
)

In [0]:
#Step 15: Standardize text data
# Remove unwanted spaces
text_columns = [
    "Employee_Name",
    "Department",
    "Designation",
    "City",
    "Employment_Status",
    "Email",
    "Manager_ID"
]

for col in text_columns:
    df[col] = df[col].str.strip()

# Standardize capitalization
df["Department"] = df["Department"].str.title()
df["Designation"] = df["Designation"].str.title()
df["City"] = df["City"].str.title()
df["Employment_Status"] = df["Employment_Status"].str.title()

# Convert emails to lowercase
df["Email"] = df["Email"].str.lower()

print("Text standardization completed!")

display(df[text_columns].head())

In [0]:
#Step 16: Convert columns into numeric values
# Columns that should contain numeric values
numeric_columns = [
    "Age",
    "Annual_Salary",
    "Performance_Score",
    "Working_Hours",
    "Projects_Completed",
    "Customer_Satisfaction",
    "Annual_Bonus",
    "Experience_Years",
    "Training_Hours"
]

# Convert each column into numeric format
for col in numeric_columns:

    # Remove currency symbols and commas from salary/bonus
    if col in ["Annual_Salary", "Annual_Bonus"]:
        df[col] = df[col].str.replace(
            r"[₹,$,\s]", "", regex=True
        )

    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Numeric conversion completed!")

# Check data types
display(df[numeric_columns].dtypes.reset_index())

In [0]:
#Step 17: Convert date columns
# Convert date columns
date_columns = [
    "Joining_Date",
    "Last_Appraisal"
]

for col in date_columns:
    df[col] = pd.to_datetime(
        df[col],
        format="%Y-%m-%d",
        errors="coerce"
    )

print("Date conversion completed!")

# Check date data types
display(df[date_columns].dtypes.reset_index())

In [0]:
#Step 18: Verify the converted data
print("Dataset information:")
df.info()

print("\nSample records:")
display(df[["Employee_ID", "Joining_Date", "Last_Appraisal"]].head())

In [0]:
#Step 19: Create data-quality flags
# 1. Missing information
df["Missing_Name"] = df["Employee_Name"].isna()
df["Missing_Department"] = df["Department"].isna()

# 2. Invalid age
df["Invalid_Age"] = (
    df["Age"].isna() |
    (df["Age"] <= 0)
)

# 3. Invalid experience
df["Invalid_Experience"] = (
    df["Experience_Years"].isna() |
    (df["Experience_Years"] < 0) |
    (df["Experience_Years"] > df["Age"] - 18)
)

# 4. Invalid salary
df["Invalid_Salary"] = (
    df["Annual_Salary"].isna() |
    (df["Annual_Salary"] <= 0)
)

# 5. Duplicate employee IDs
df["Duplicate_Employee_ID"] = (
    df.duplicated(
        subset=["Employee_ID"],
        keep=False
    )
)

# 6. Count all quality issues per employee record
quality_columns = [
    "Missing_Name",
    "Missing_Department",
    "Invalid_Age",
    "Invalid_Experience",
    "Invalid_Salary",
    "Invalid_Joining_Date",
    "Duplicate_Employee_ID"
]

df["Total_Data_Issues"] = df[quality_columns].sum(axis=1)

print("Data quality flags created successfully!")

print("Records with at least one issue:",
      (df["Total_Data_Issues"] > 0).sum())

display(df[quality_columns + ["Total_Data_Issues"]].head(10))


In [0]:
#Step 20: Handle missing values
# Fill missing department values
df["Department"] = df["Department"].fillna("Unknown")

# Keep missing employee names as NaN
# Missing_Name flag already identifies these records

print("Missing values handled!")

print("Missing Employee Names:", df["Employee_Name"].isna().sum())
print("Missing Departments:", df["Department"].isna().sum())

display(
    df[df["Missing_Name"] | df["Missing_Department"]][
        ["Employee_ID", "Employee_Name",
         "Department", "Missing_Name", "Missing_Department"]
    ].head(10)
)

In [0]:
#Step 21: Handle duplicate employee records
# Extract duplicate records for HR investigation
duplicate_review = df[
    df["Duplicate_Employee_ID"] == True
].copy()

# Count duplicate employee IDs
duplicate_id_count = (
    duplicate_review["Employee_ID"].nunique()
)

# Count extra duplicate records
extra_duplicate_count = df.duplicated(
    subset=["Employee_ID"],
    keep="first"
).sum()

print("Duplicate Employee IDs:", duplicate_id_count)
print("Extra Duplicate Records:", extra_duplicate_count)
print("Records retained in main dataset:", len(df))

# Display duplicate records for investigation
display(
    duplicate_review[
        ["Employee_ID", "Employee_Name",
         "Department", "Designation",
         "Duplicate_Employee_ID"]
    ].head(10)
)

In [0]:
#Step 22: Save cleaned data into the Silver layer
# Convert cleaned Pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Save into Silver layer as a Delta table
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("hr_catalog.silver.clean_employees")

print("Cleaned HR dataset saved successfully!")

In [0]:
%sql
/* Step 22.1: Verify the Silver table */


SELECT *
FROM hr_catalog.silver.clean_employees
LIMIT 10;


In [0]:
%sql
/*Step 23: Department-wise employee count and salary*/

SELECT
    Department,
    COUNT(*) AS Total_Employee_Records,
    COUNT(DISTINCT Employee_ID) AS Unique_Employee_IDs,

    SUM(
        CASE WHEN Invalid_Salary = FALSE
        THEN Annual_Salary ELSE 0 END
    ) AS Total_Annual_Salary,

    ROUND(
        AVG(
            CASE WHEN Invalid_Salary = FALSE
            THEN Annual_Salary END
        ), 2
    ) AS Average_Salary,

    MIN(
        CASE WHEN Invalid_Salary = FALSE
        THEN Annual_Salary END
    ) AS Minimum_Salary,

    MAX(
        CASE WHEN Invalid_Salary = FALSE
        THEN Annual_Salary END
    ) AS Maximum_Salary

FROM hr_catalog.silver.clean_employees

GROUP BY Department
ORDER BY Total_Annual_Salary DESC;

In [0]:
%sql
/*Step 24: Department-wise performance analysis*/

SELECT
    Department,

    COUNT(*) AS Total_Employee_Records,

    ROUND(AVG(Performance_Score), 2)
        AS Average_Performance,

    ROUND(AVG(Customer_Satisfaction), 2)
        AS Average_Customer_Satisfaction,

    SUM(Projects_Completed)
        AS Total_Projects_Completed,

    ROUND(AVG(Training_Hours), 2)
        AS Average_Training_Hours

FROM hr_catalog.silver.clean_employees

GROUP BY Department

ORDER BY Average_Performance DESC;

In [0]:
%sql
-- Step 24: View the performance results--
SELECT
    Department,
    ROUND(AVG(Performance_Score), 2) AS Avg_Performance,
    ROUND(AVG(Customer_Satisfaction), 2) AS Avg_Customer_Satisfaction,
    SUM(Projects_Completed) AS Total_Projects

FROM hr_catalog.silver.clean_employees

GROUP BY Department

ORDER BY Avg_Performance DESC;


In [0]:
%sql
--Step 25: Employee status and resignation analysis--

SELECT
    Department,

    COUNT(*) AS Total_Records,

    SUM(CASE WHEN Employment_Status = 'Active'
        THEN 1 ELSE 0 END) AS Active_Employees,

    SUM(CASE WHEN Employment_Status = 'Resigned'
        THEN 1 ELSE 0 END) AS Resigned_Employees,

    SUM(CASE WHEN Employment_Status = 'Inactive'
        THEN 1 ELSE 0 END) AS Inactive_Employees,

    SUM(CASE WHEN Employment_Status = 'On Leave'
        THEN 1 ELSE 0 END) AS On_Leave_Employees

FROM hr_catalog.silver.clean_employees

GROUP BY Department

ORDER BY Resigned_Employees DESC;

In [0]:
%sql

SELECT
    Employee_ID,
    Employee_Name,
    Department,
    Designation,
    Manager_ID,
    Employment_Status

FROM hr_catalog.silver.clean_employees

ORDER BY Manager_ID

LIMIT 50;

In [0]:
%sql

SELECT
    Manager_ID,
    COUNT(*) AS Employee_Records
FROM hr_catalog.silver.clean_employees
GROUP BY Manager_ID
ORDER BY Employee_Records DESC
LIMIT 20;

In [0]:
%sql

SELECT
    COUNT(*) AS Total_Records,

    SUM(CASE WHEN Missing_Name THEN 1 ELSE 0 END)
        AS Missing_Names,

    SUM(CASE WHEN Missing_Department THEN 1 ELSE 0 END)
        AS Missing_Departments,

    SUM(CASE WHEN Invalid_Age THEN 1 ELSE 0 END)
        AS Invalid_Ages,

    SUM(CASE WHEN Invalid_Experience THEN 1 ELSE 0 END)
        AS Invalid_Experience,

    SUM(CASE WHEN Invalid_Salary THEN 1 ELSE 0 END)
        AS Invalid_Salaries,

    SUM(CASE WHEN Invalid_Joining_Date THEN 1 ELSE 0 END)
        AS Future_Joining_Dates,

    SUM(CASE WHEN Duplicate_Employee_ID THEN 1 ELSE 0 END)
        AS Duplicate_Records,

    SUM(CASE WHEN Total_Data_Issues > 0 THEN 1 ELSE 0 END)
        AS Records_With_Issues,

    ROUND(
        100.0 * SUM(
            CASE WHEN Total_Data_Issues = 0
            THEN 1 ELSE 0 END
        ) / COUNT(*), 2
    ) AS Data_Quality_Percentage

FROM hr_catalog.silver.clean_employees;

In [0]:
%sql
/* #create a gold department summary table */


CREATE OR REPLACE TABLE hr_catalog.gold.department_hr_summary AS

SELECT
    Department,

    COUNT(*) AS Total_Employee_Records,

    COUNT(DISTINCT Employee_ID) AS Unique_Employee_IDs,

    SUM(CASE WHEN Employment_Status = 'Active'
        THEN 1 ELSE 0 END) AS Active_Employees,

    SUM(CASE WHEN Employment_Status = 'Resigned'
        THEN 1 ELSE 0 END) AS Resigned_Employees,

    SUM(CASE WHEN Employment_Status = 'Inactive'
        THEN 1 ELSE 0 END) AS Inactive_Employees,

    SUM(CASE WHEN Employment_Status = 'On Leave'
        THEN 1 ELSE 0 END) AS On_Leave_Employees,

    SUM(CASE WHEN Invalid_Salary = FALSE
        THEN Annual_Salary ELSE 0 END) AS Total_Annual_Salary,

    ROUND(AVG(
        CASE WHEN Invalid_Salary = FALSE
        THEN Annual_Salary END
    ), 2) AS Average_Salary,

    ROUND(AVG(Performance_Score), 2)
        AS Average_Performance,

    SUM(Projects_Completed)
        AS Total_Projects_Completed,

    SUM(Total_Data_Issues)
        AS Total_Data_Quality_Issues

FROM hr_catalog.silver.clean_employees

GROUP BY Department;

In [0]:
%sql
--Step 29: Verify the Gold table--
SELECT *
FROM hr_catalog.gold.department_hr_summary
ORDER BY Total_Employee_Records DESC;

In [0]:
%sql
--Step 30: Create Gold data-quality summary--
CREATE OR REPLACE TABLE hr_catalog.gold.hr_data_quality AS

SELECT
    COUNT(*) AS Total_Records,

    SUM(CASE WHEN Missing_Name
        THEN 1 ELSE 0 END) AS Missing_Names,

    SUM(CASE WHEN Missing_Department
        THEN 1 ELSE 0 END) AS Missing_Departments,

    SUM(CASE WHEN Invalid_Age
        THEN 1 ELSE 0 END) AS Invalid_Ages,

    SUM(CASE WHEN Invalid_Experience
        THEN 1 ELSE 0 END) AS Invalid_Experience,

    SUM(CASE WHEN Invalid_Salary
        THEN 1 ELSE 0 END) AS Invalid_Salaries,

    SUM(CASE WHEN Invalid_Joining_Date
        THEN 1 ELSE 0 END) AS Future_Joining_Dates,

    SUM(CASE WHEN Duplicate_Employee_ID
        THEN 1 ELSE 0 END) AS Duplicate_Records,

    SUM(CASE WHEN Total_Data_Issues > 0
        THEN 1 ELSE 0 END) AS Records_With_Issues,

    ROUND(
        100.0 * SUM(
            CASE WHEN Total_Data_Issues = 0
            THEN 1 ELSE 0 END
        ) / COUNT(*), 2
    ) AS Data_Quality_Percentage

FROM hr_catalog.silver.clean_employees;

In [0]:
%sql
--Step 31: View the final quality report--
SELECT *
FROM hr_catalog.gold.hr_data_quality;

In [0]:
%sql

CREATE OR REPLACE TABLE hr_catalog.gold.employee_reporting AS

SELECT
    Employee_ID,
    Employee_Name,
    Department,
    Designation,
    Manager_ID,
    Employment_Status,
    Joining_Date,
    Experience_Years

FROM hr_catalog.silver.clean_employees;

In [0]:
%sql

SELECT *
FROM hr_catalog.gold.employee_reporting
LIMIT 20;